# Prompt Ablation Experiment
Bu notebook prompt varyantlarini tek model uzerinde karsilastirir.


In [ ]:
import subprocess
import sys
from pathlib import Path

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/LLMComparison'),
    Path('/content/drive/MyDrive/LLMComparison'),
]
PROJECT_ROOT = next((root for root in candidate_roots if (root / 'experiments').exists() and (root / 'src').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Project root not found. Open the repo or mount Drive first.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
MODEL = 'qwen2-vl'
PROMPTS = ['baseline', 'detailed', 'structured', 'chain_of_thought']
NUM_SAMPLES = 50
OUTPUT_PATH = PROJECT_ROOT / 'results' / 'prompt_ablation.json'
DEBUG = False
print(f'Model: {MODEL}')
print(f'Output: {OUTPUT_PATH}')

In [ ]:
import time

command = [
    sys.executable,
    str(PROJECT_ROOT / 'experiments' / 'run_prompt_ablation.py'),
    '--model', MODEL,
    '--prompts', *PROMPTS,
    '--num-samples', str(NUM_SAMPLES),
    '--output', str(OUTPUT_PATH),
]
if DEBUG:
    command.append('--debug')

print('Running command:')
print(command)

total_steps = max(1, len(PROMPTS))
completed_steps = 0
start_ts = time.time()

process = subprocess.Popen(
    command,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    text_line = line.rstrip()
    if text_line.startswith('Testing prompt:'):
        completed_steps += 1
    elapsed = time.time() - start_ts
    elapsed_min = elapsed / 60.0
    if completed_steps > 0:
        avg_per_step = elapsed / completed_steps
        remaining_steps = max(0, total_steps - completed_steps)
        eta_sec = avg_per_step * remaining_steps
        eta_min = eta_sec / 60.0
        print(f'[{completed_steps}/{total_steps}] elapsed={elapsed_min:.1f}m eta~{eta_min:.1f}m | {text_line}')
    else:
        print(f'[0/{total_steps}] elapsed={elapsed_min:.1f}m eta~unknown | {text_line}')

return_code = process.wait()
total_elapsed_min = (time.time() - start_ts) / 60.0
print(f'Finished with code={return_code} in {total_elapsed_min:.1f}m')

if return_code != 0:
    raise RuntimeError(f'run_prompt_ablation failed with exit code {return_code}')

In [ ]:
import json
import pandas as pd

if OUTPUT_PATH.exists():
    payload = json.loads(OUTPUT_PATH.read_text())
    rows = [
        {'prompt_name': prompt_name, **scores}
        for prompt_name, scores in payload.get('results', {}).items()
    ]
    display(pd.DataFrame(rows))
else:
    print('Output file not found yet.')